<a href="https://colab.research.google.com/github/fabrizzio2901/Challenge-Telecom-X-an-lisis-de-evasi-n-de-clientes---Parte-2/blob/main/Challenge_Telecom_X_an%C3%A1lisis_de_evasi%C3%B3n_de_clientes_Parte_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#🛠️ Preparación de los Datos

In [2]:
import pandas as pd

# Cargamos el archivo
df_telecom = pd.read_csv('datos_tratados.csv')

# Vemos que todo esté en orden
df_telecom.head()

,customerID,Churn,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,Charges.Monthly,Charges.Total,Cuentas_Diarias
0,0002-ORFBO,no,female,0,yes,yes,9,yes,no,dsl,...,no,yes,yes,no,one year,yes,mailed check,65.6,593.30,2.19
1,0003-MKNFE,no,male,0,no,no,9,yes,yes,dsl,...,no,no,no,yes,month-to-month,no,mailed check,59.9,542.40,2.00
2,0004-TLHLJ,yes,male,0,no,no,4,yes,no,fiber optic,...,yes,no,no,no,month-to-month,yes,electronic check,73.9,280.85,2.46
3,0011-IGKFF,yes,male,1,yes,no,13,yes,no,fiber optic,...,yes,no,yes,yes,month-to-month,yes,electronic check,98.0,1237.85,3.27
4,0013-EXCHZ,yes,female,1,yes,no,3,yes,no,fiber optic,...,no,yes,yes,no,month-to-month,yes,mailed check,83.9,267.40,2.80


In [4]:
# Quitamos la columna 'customerID'
df_modelo = df_telecom.drop('customerID', axis=1)

# Confirmamos que ya no está
print(df_modelo.columns.tolist())
df_modelo.head()

['Churn', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'Charges.Monthly', 'Charges.Total', 'Cuentas_Diarias']


,Churn,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,Charges.Monthly,Charges.Total,Cuentas_Diarias
0,no,female,0,yes,yes,9,yes,no,dsl,no,...,no,yes,yes,no,one year,yes,mailed check,65.6,593.30,2.19
1,no,male,0,no,no,9,yes,yes,dsl,no,...,no,no,no,yes,month-to-month,no,mailed check,59.9,542.40,2.00
2,yes,male,0,no,no,4,yes,no,fiber optic,no,...,yes,no,no,no,month-to-month,yes,electronic check,73.9,280.85,2.46
3,yes,male,1,yes,no,13,yes,no,fiber optic,no,...,yes,no,yes,yes,month-to-month,yes,electronic check,98.0,1237.85,3.27
4,yes,female,1,yes,no,3,yes,no,fiber optic,no,...,no,yes,yes,no,month-to-month,yes,mailed check,83.9,267.40,2.80


In [5]:
# Convertimos el Churn (nuestra meta) en números: yes = 1, no = 0
# Usamos .map
df_modelo['Churn'] = df_modelo['Churn'].map({'yes': 1, 'no': 0})

# 3. Transformamos todas las demás palabras en columnas de 0 y 1
df_final = pd.get_dummies(df_modelo, drop_first=True)

# 4. Convertimos todo a números enteros para que la tabla sea fácil de leer
df_final = df_final.astype(int)

# Vemos el resultado final
print("¡Transformación completada! Ahora todo es lenguaje matemático.")
df_final.head()

¡Transformación completada! Ahora todo es lenguaje matemático.


,Churn,SeniorCitizen,tenure,Charges.Monthly,Charges.Total,Cuentas_Diarias,gender_male,Partner_yes,Dependents_yes,PhoneService_yes,...,DeviceProtection_yes,TechSupport_yes,StreamingTV_yes,StreamingMovies_yes,Contract_one year,Contract_two year,PaperlessBilling_yes,PaymentMethod_credit card (automatic),PaymentMethod_electronic check,PaymentMethod_mailed check
0,0,0,9,65,593,2,0,1,1,1,...,0,1,1,0,1,0,1,0,0,1
1,0,0,9,59,542,2,1,0,0,1,...,0,0,0,1,0,0,0,0,0,1
2,1,0,4,73,280,2,1,0,0,1,...,1,0,0,0,0,0,1,0,1,0
3,1,1,13,98,1237,3,1,1,0,1,...,1,0,1,1,0,0,1,0,1,0
4,1,1,3,83,267,2,0,1,0,1,...,0,1,1,0,0,0,1,0,0,1


In [9]:
# Calculamos el conteo y el porcentaje de Churn
conteo_churn = df_telecom['Churn'].value_counts()
porcentaje_churn = df_telecom['Churn'].value_counts(normalize=True) * 100

print("--- Análisis de Balance de Clases ---")
print(f"Clientes que se quedan (No): {conteo_churn['no']} ({porcentaje_churn['no']:.2f}%)")
print(f"Clientes que se van (Yes): {conteo_churn['yes']} ({porcentaje_churn['yes']:.2f}%)")

if porcentaje_churn['yes'] < 40:
    print("\n Existe un desbalance de clases.")
else:
    print("\n Las clases están relativamente balanceadas.")

--- Análisis de Balance de Clases ---
Clientes que se quedan (No): 5174 (73.46%)
Clientes que se van (Yes): 1869 (26.54%)

 Existe un desbalance de clases.


In [11]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# Separamos los datos:
# 'X' son las pistas (todo menos Churn)
# 'y' es la respuesta (solo Churn)
X = df_final.drop('Churn', axis=1)
y = df_final['Churn']

# Dividimos en Entrenamiento (70%) y Prueba (30%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Aplicamos SMOTE para balancear solo el entrenamiento
# Esto crea los clientes "simulados"
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

# Verificamos cómo quedaron los datos ahora
print("--- Antes del balanceo (Entrenamiento) ---")
print(y_train.value_counts())

print("\n--- Después del balanceo (Entrenamiento) ---")
print(y_train_bal.value_counts())

--- Antes del balanceo (Entrenamiento) ---
Churn
0    3612
1    1318
Name: count, dtype: int64

--- Después del balanceo (Entrenamiento) ---
Churn
1    3612
0    3612
Name: count, dtype: int64


In [13]:
from sklearn.preprocessing import StandardScaler

# Creamos el objeto escalador
scaler = StandardScaler()

# "Entrenamos" el escalador con los datos de entrenamiento balanceados
# Esto calcula la media y desviación para ajustar los números
X_train_scaled = scaler.fit_transform(X_train_bal)

# Aplicamos la misma transformación a los datos de prueba
X_test_scaled = scaler.transform(X_test)

# Convertimos a DataFrame para que se vea ordenado (opcional)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)
X_train_scaled.head()

,SeniorCitizen,tenure,Charges.Monthly,Charges.Total,Cuentas_Diarias,gender_male,Partner_yes,Dependents_yes,PhoneService_yes,MultipleLines_yes,...,DeviceProtection_yes,TechSupport_yes,StreamingTV_yes,StreamingMovies_yes,Contract_one year,Contract_two year,PaperlessBilling_yes,PaymentMethod_credit card (automatic),PaymentMethod_electronic check,PaymentMethod_mailed check
0,-0.403622,-0.899703,0.815379,-0.691316,1.252005,1.146579,1.271767,-0.5203,0.347162,-0.789307,...,1.609490,-0.512559,-0.742213,1.326805,-0.423135,-0.445653,0.811889,-0.436472,1.366677,-0.460301
1,-0.403622,0.308421,0.254672,0.271253,0.289621,1.146579,-0.786308,-0.5203,0.347162,1.266934,...,-0.621315,-0.512559,-0.742213,-0.753690,-0.423135,-0.445653,-1.231695,2.291099,-0.731702,-0.460301
2,-0.403622,-0.358131,-1.707802,-0.763566,-1.635147,1.146579,-0.786308,-0.5203,0.347162,-0.789307,...,-0.621315,-0.512559,-0.742213,-0.753690,-0.423135,2.243900,0.811889,-0.436472,-0.731702,2.172490
3,-0.403622,-0.233152,0.359804,-0.202486,0.289621,1.146579,-0.786308,-0.5203,0.347162,1.266934,...,-0.621315,-0.512559,-0.742213,-0.753690,-0.423135,-0.445653,0.811889,-0.436472,-0.731702,2.172490
4,-0.403622,0.016804,-0.270991,-0.117890,0.289621,1.146579,1.271767,-0.5203,0.347162,-0.789307,...,-0.621315,1.950996,-0.742213,-0.753690,-0.423135,-0.445653,-1.231695,-0.436472,-0.731702,2.172490
